In [49]:
# Important library
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_community.document_loaders import DirectoryLoader

import fitz  # PyMuPDF
import os

from langchain_text_splitters import RecursiveCharacterTextSplitter

import numpy as np
from sentence_transformers import SentenceTransformer
import ollama 

import uuid
import chromadb

from ollama import Client
from ollama import chat

In [54]:
# Section 1: Document Loader
# load all the pdf pages from all the pdf file from the path

def Read_PDF_Document(path):
    dir_loader=DirectoryLoader(path,
        glob="**/*.pdf", ## Pattern to match files  
        loader_cls= PyMuPDFLoader, ##loader class to use
        show_progress=False)
    pdf_documents=dir_loader.load()
    return pdf_documents

path = "../RAG LLM document to response/Document"
pdf_documents = READ_PDF_Document(path)

In [27]:
# Section 2: Extract information from the pdf files
# i).  Extracting Raw text (page content, metadata)
# ii). Extracting Image (chart, model) and saving in the image folder in .jpeg format
# iii). Extracting Table and storing in the Table folder in .csv format

In [21]:
# i). Extracting the Raw text (page content, metadata)
# Text splitter using the default RecursiveCharacterTextSplitter with overlap of 10-20% of token size

def Extract_Split_Documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
    
    return split_docs

chuncks = Extract_Split_Documents(pdf_documents,chunk_size=1000,chunk_overlap=200)

Split 562 documents into 853 chunks

Example chunk:
Content: www.allitebooks.com...


In [26]:
# ii). Extracting Image (chart, model) and saving in the image folder in .jpeg format
# It will extract and save all the image into the image folder 

def Extract_Images(pdf_path, output_path):
    pdf = fitz.open(pdf_path)

    os.makedirs(output_path, exist_ok=True)

    for page_num in range(len(pdf)):
        page = pdf[page_num]

        for img_index, img in enumerate(page.get_images(full=True)):
            xref = img[0]

            base_image = pdf.extract_image(xref)
            image_bytes = base_image["image"]
            image_ext = base_image["ext"]

            image_name = f"page_{page_num+1}_{img_index}.{image_ext}"

            with open(os.path.join(output_path, image_name),"wb") as f:
                f.write(image_bytes)

    pdf.close()

pdf_path = "../RAG LLM document to response/Document/Practical Statistics for Data Scientists.pdf"
output_path = "../RAG LLM document to response/images"
img = Extract_Images(pdf_path, output_path)

In [28]:
# iii). Extracting Table and storing in the Table folder in .csv format
# Note: This pdf is in the image text based format so unable to perform table extraction

# import camelot

# def Extract_table(doc_path,csv_path):
#     tables = camelot.read_pdf(doc_path, pages="all")

#     for i, table in enumerate(tables):
#         df = table.df
#         df.to_csv(f"{csv_path}table_{i}.csv", index=False)

# doc_path = "../RAG LLM document to response/Document/Practical Statistics for Data Scientists.pdf"
# csv_path =  "../RAG LLM document to response/Tables/"

# # table = Extract_table(doc_path,csv_path)

In [50]:
# Section 3: Creation of Vector Embedding 
# i). Page content (raw text) --> Embedding Model --> Vector Embedding of dimension :768
# ii). Images to text using llava:7b --> Embedding Model --> Vector Embedding of dimension :768
# iii). Table to text (raw text) --> Embedding Model --> Vector Embedding of dimension :768
# Note: Assemebling each embedding page wise then storing in the Vector DB (chroma DB)

In [30]:
# i). Page content (raw text) --> Embedding Model --> Vector Embedding of dimension :768
# Load Embedding Model ("embeddinggemma:300m") + Generate Embeddings

def Generate_Embeddings(model_name, documents):
    # Extract text from Document objects
    embed_list = list()
    for doc in documents:
        text = doc.page_content
        embeddings = ollama.embed(model_name, text)
        embed_list.append(embeddings.embeddings)

    return embed_list

# Model Name
model_name = "embeddinggemma:300m"

# Generate embeddings
embeddings = Generate_Embeddings(model_name, chuncks)

# print("Embeddings Shape:", len(embeddings))
# print("Embeddings type:", type(embeddings))

In [55]:
# ii). Images to text using llava:7b --> Embedding Model --> Vector Embedding of dimension :768
# Now converting all the image into the vector embedding using the ollama --> llava multimodel

def Image_text(img_path):
    response = ollama.chat(model="llava:7b",
        messages=[{"role": "user",
            "content": "Describe this image in detail of 100 words.",
            "images": [img_path]}])

    caption = response["message"]["content"]
    return caption
    
img_path = "../RAG LLM document to response/images/page_114_0.jpeg"
print(Image_text(img_path))

 The image presents a series of three bar graphs, each representing different data sets with the x-axis label "Income" ranging from $0 to $5,000. 

The first graph is the shortest and has the smallest number of bars on it compared to the other two graphs. Despite its size, it conveys a message. The vertical axis labels each bar with a numerical value ranging from -100 to 300, indicating how many individuals in that income bracket belong to a certain category. 

The second graph is taller than the first and features a larger number of bars. These bars are colored in shades of blue, green, and purple, each representing a different category within the income brackets. The bars vary in length, suggesting a more diverse representation of the data.

The third graph is the tallest and has even more bars than the second. It too uses a color-coding system, but this time the colors are red, yellow, and orange. Like the other two graphs, the bars' lengths correspond to the number of individuals i

In [56]:
# iii). Table to text (raw text) --> Embedding Model --> Vector Embedding of dimension :768
# using the embedding model to conver table data into vector embedding

In [33]:
# Note: As per the document images and table folder data is not significant to procude any inference
# So proceeding further in the next step 

In [34]:
# Section 4: Consolidating the Text, Image and Table Embeddings 
# Page wise division with Toeken length an overlao size

In [35]:
# Section 5: Initialize and storing the Vector DB keeping data persistency
# storing locally in vectordb (chromadb) 
# Using uuid for unique value and avoiding duplicate value to store in chromaDB

def Initialize_Vector_Store(collection_name="pdf_documents", persist_directory="../data/vector_store"):
    # Create folder if not present
    os.makedirs(persist_directory, exist_ok=True)
    
    # Create persistent ChromaDB client
    client = chromadb.PersistentClient(path=persist_directory)
    client.delete_collection(name=collection_name)

    # Create or load collection
    collection = client.get_or_create_collection(
        name=collection_name,
        metadata={ "description": "PDF document embeddings for RAG" }
    )

    print(f"Collection Name: {collection_name}")
    print(f"Existing Documents: {collection.count()}")

    return collection

# Add Documents + Embeddings
def Add_Documents_to_VectorStore(collection, documents, embeddings):

    # Empty lists for storage
    ids = []
    metadatas = []
    document_texts = []
    embedding_list = []

    # Loop through each document
    for i, (doc, embedding) in enumerate(zip(documents, embeddings) ):
        # Generate Unique ID
        doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
        ids.append(doc_id)

        # Metadata
        metadata = dict(doc.metadata)
        metadata["doc_index"] = i
        metadata["content_length"] = len(doc.page_content)
        metadatas.append(metadata)

        # Store document text
        document_texts.append(doc.page_content)

        # Store embedding
        embedding_list.append(embedding[0])

    # Add data into ChromaDB        
    collection.add(
        ids=ids,
        documents=document_texts,
        metadatas=metadatas,
        embeddings=embedding_list
    )

    print("Documents added successfully")

    print("Total Documents:", collection.count())

collection = Initialize_Vector_Store()
Add_Documents_to_VectorStore(collection, chuncks, embeddings)   

Collection Name: pdf_documents
Existing Documents: 0
Documents added successfully
Total Documents: 853


In [42]:
# Section 6: Retrieval relevant Document from vector DB based on semantic search 
# Firstly converting user query into vectors to search in the Chroma DB (local vector store)
# Retrieve Similar Documents based on the cosine similarity

# There are two methods to Retrieve Similar Documents:
# i). Noraml method based on the cosine similarity using semantic search
# ii). FAISS for the fast accurate and efficient result

def Retrieve_Relevant_Documents(query, collection, top_k, score_threshold=0.0):
    
    model_name = "embeddinggemma:300m"
    query_embedding = ollama.embed(model_name, query)
    
    results = collection.query(query_embeddings=query_embedding.embeddings, n_results=top_k)
    
    return results

# query = "what is machine learning?"
# results = Retrieve_Relavent_Documents(query, collection, top_k=3)

In [39]:
query="Application of Machine learning in real world"

embedding_result = Retrieve_Relevant_Documents(query, collection, top_k=5, score_threshold=0.0)

context = embedding_result['documents']
print(context)
metadata = embedding_result['metadatas'] 
print(metadata)

[['Confusion Matrix\nThe Rare Class Problem\nPrecision, Recall, and Specificity\nROC Curve\nAUC\nLift\nFurther Reading\nStrategies for Imbalanced Data\nUndersampling\nOversampling and Up/Down Weighting\nData Generation\nCost-Based Classification\nExploring the Predictions\nFurther Reading\nSummary\n6. Statistical Machine Learning\nK-Nearest Neighbors\nA Small Example: Predicting Loan Default\nDistance Metrics\nOne Hot Encoder\nStandardization (Normalization, Z-Scores)\nChoosing K', 'Figure 6-9. The predicted outcomes from XGBoost applied to the loan default data', 't-tests, t-Tests-Further Reading\nstatistical inference, classical inference pipeline, Statistical Experiments and\nSignificance Testing\nstatistical machine learning, Statistical Machine Learning-Summary\nbagging and the random forest, Bagging and the Random Forest-\nHyperparameters\nboosting, Boosting-Summary\navoiding overfitting using regularization, Regularization: Avoiding\nOverfitting\nhyperparameters and cross-valida

In [57]:
# Section 7: For reranking the retrieval document using reranker LLM model 
# Using user query and chucnk of similar document based on the 
# Re ranking of the retreive document


In [51]:
# Section 8: Generation of response from the LLM model 
# by passing ranked relevant data from Vector DB

query = "what is K-Means Clustering and it usecase"

embedding_result = Retrieve_Relevant_Documents(query, collection, top_k=5, score_threshold=0.0)

# Creating Prompt to generate response from LLM by setting some ground rules
prompt = f"""
You are a helpful AI assistant.

Use ONLY the provided context to answer the question.

Context:
{context}

Question:
{query}

Metadata:
{metadata}

Answer:
"""

model_name = 'gemma3:latest'

def Generating_Response_LLM(query,embedding_result,model_name):
    context = embedding_result['documents']
    metadata = embedding_result['metadatas'] 

    # Generate Response from Gemma3
    stream = chat(
        model = model_name,
        messages=[
            {
                'role': 'user',
                'content': prompt
            }
        ],
        stream=True
    )
    
    for chunk in stream:
        print(chunk['message']['content'], end='', flush=True)

# Calling  function
Generating_Response_LLM(query,embedding_result,model_name)

The provided context does not contain information about K-Means Clustering or its use cases. It primarily discusses K-Nearest Neighbors (KNN) and statistical machine learning techniques like decision trees, bagging, boosting, regularization, and XGBoost, focusing on a loan default prediction example.

In [52]:
# Section :9 
# i). Config file setup (A/B testing, model upgradation), 
# ii). Setting up the Gaudrails of the model

In [53]:
# Section : 10 Observability and evaluation metrics
# i).  Hallucations
# ii). Limitated information or repetative knowledge , bias, null, duplicate
# ii). Imporper chuncking of the document
# Evaluation
# i). Context precision
# ii). Context Recall
# iii). Faithfulness
# iv). Answer relevancy and correctness